# Merge BioMistral adapter and quantize to GGUF (Q4_K_M)
Settings before running: **Accelerator = None** (CPU is enough), **Internet = On**.
Intermediate files go to `/kaggle/temp` (large, not saved). Only the final file goes to `/kaggle/working`.

## 1. Setup

In [1]:
!pip install -q peft accelerate huggingface_hub sentencepiece protobuf gguf

import os
os.environ["HF_HOME"] = "/kaggle/temp/hf"   # keep the 14 GB download off /kaggle/working

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.5/118.5 kB 2.5 MB/s eta 0:00:00


## 2. Find your adapter folder
Copy the folder that contains `adapter_config.json`.

In [2]:
!find /kaggle/input -wholename "/kaggle/input/datasets/satyamvishwakarma06/biomistral/adapter_config.json"

/kaggle/input/datasets/satyamvishwakarma06/biomistral/adapter_config.json


## 3. Merge the adapter into the base model (CPU)

In [3]:
pip install "torchao>=0.16.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 42.7 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0
Note: you may need to restart the kernel to use updated packages.


In [4]:
import gc
import torch
from huggingface_hub import hf_hub_download
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

BASE_MODEL = "BioMistral/BioMistral-7B"
ADAPTER_DIR = "/kaggle/input/datasets/satyamvishwakarma06/biomistral"   # <-- from step 2
MERGED_DIR = "/kaggle/temp/merged"

base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, torch_dtype=torch.float16, device_map="cpu", low_cpu_mem_usage=True
)
model = PeftModel.from_pretrained(base, ADAPTER_DIR).merge_and_unload()
model.save_pretrained(MERGED_DIR, safe_serialization=True)

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(MERGED_DIR)
try:
    hf_hub_download(BASE_MODEL, "tokenizer.model", local_dir=MERGED_DIR)
except Exception as err:
    print("Skipped tokenizer.model:", err)

del model, base
gc.collect()

Failed to load /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /usr/local/lib/python3.13/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.13/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so


config.json:   0%|          | 0.00/567 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 14.5GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

...

/usr/local/lib/python3.13/dist-packages/peft/config.py:220: UserWarning: Unexpected keyword arguments ['kasa_config'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  493kB            

tokenizer.model: downloading bytes:           |  0.00B            

403

In [5]:
# Free disk: the base model download is no longer needed
!rm -rf /kaggle/temp/hf

## 4. Convert to GGUF (fp16)

In [6]:
!git clone -q --depth 1 https://github.com/ggml-org/llama.cpp /kaggle/temp/llama.cpp
!python /kaggle/temp/llama.cpp/convert_hf_to_gguf.py /kaggle/temp/merged --outfile /kaggle/temp/biomistral-f16.gguf --outtype f16
!rm -rf /kaggle/temp/merged

INFO:hf-to-gguf:Loading model: merged
INFO:numexpr.utils:NumExpr defaulting to 4 threads.
INFO:hf-to-gguf:Model architecture: MistralForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:output.weight,               torch.float16 --> F16, shape = {4096, 32000}
INFO:hf-to-gguf:token_embd.weight,           torch.float16 --> F16, shape = {4096, 32000}
INFO:hf-to-gguf:blk.0.attn_norm.weight,      torch.float16 --> F32, shape = {4096}
INFO:hf-to-gguf:blk.0.ffn_down.weight,       torch.float16 --> F16, shape = {14336, 4096}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,       torch.float16 --> F16, shape = {4096, 14336}
INFO:hf-to-gguf:blk.0.ffn_up.weight,         torch.float16 --> F16, shape = {4096, 14336}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,       torch.float16 --> F32, shape = {4096}
INFO:hf-to-gguf:blk.0.attn_k.weight,         torch.float16 --> F16, shape

## 5. Build the quantizer and make the 4-bit file

In [7]:
!cd /kaggle/temp/llama.cpp && cmake -B build -DGGML_CUDA=OFF -DLLAMA_CURL=OFF > /dev/null && cmake --build build --target llama-quantize -j 4

CMAKE_BUILD_TYPE=Release
[  0%] Building CXX object ggml/src/CMakeFiles/ggml-base.dir/ggml.cpp.o
[  0%] Building C object ggml/src/CMakeFiles/ggml-base.dir/ggml.c.o
[  0%] Building CXX object common/CMakeFiles/llama-common-base.dir/build-info.cpp.o
[  0%] Building CXX object vendor/cpp-httplib/CMakeFiles/cpp-httplib.dir/httplib.cpp.o
[  0%] Building C object ggml/src/CMakeFiles/ggml-base.dir/ggml-alloc.c.o
[  0%] Linking CXX static library libllama-common-base.a
[  0%] Built target llama-common-base
[  3%] Building CXX object ggml/src/CMakeFiles/ggml-base.dir/ggml-backend.cpp.o
[  3%] Building CXX object ggml/src/CMakeFiles/ggml-base.dir/ggml-backend-meta.cpp.o
[  3%] Building CXX object ggml/src/CMakeFiles/ggml-base.dir/ggml-opt.cpp.o
[  3%] Building CXX object ggml/src/CMakeFiles/ggml-base.dir/ggml-threading.cpp.o
[  6%] Building C object ggml/src/CMakeFiles/ggml-base.dir/ggml-quants.c.o
[  6%] Building CXX object ggml/src/CMakeFiles/ggml-base.dir/gguf.cpp.o
[  6%] Linking CXX shared

In [8]:
!/kaggle/temp/llama.cpp/build/bin/llama-quantize /kaggle/temp/biomistral-f16.gguf /kaggle/working/biomistral-med-q4_k_m.gguf Q4_K_M
!ls -lh /kaggle/working

version: 0.5.0-dev (build 1, commit 1537a0a)
built with GNU 13.3.0 for Linux x86_64
llama_quantize: quantizing '/kaggle/temp/biomistral-f16.gguf' to '/kaggle/working/biomistral-med-q4_k_m.gguf' as Q4_K_M
llama_model_loader: loaded meta data with 30 key-value pairs and 291 tensors from /kaggle/temp/biomistral-f16.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Merged
llama_model_loader: - kv   3:                         general.size_label str              = 7.2B
llama_model_loader: - kv   4:                          llama.block_count u32              = 32
llama_model_loader: - kv   5:                       llama.context_len

In [9]:
from IPython.display import FileLink

# Pass the filename or relative path of your file
FileLink("/kaggle/working/biomistral-med-q4_k_m.gguf")


/kaggle/working/biomistral-med-q4_k_m.gguf